# CME Futures: Double Machine Learning

This notebook estimates the configured treatment effect for each return horizon. The request
resolves the outcome, treatment, confounders, timing, embargo, nuisance models, and placebo design
before fitting. Missing confounders are an error rather than an invitation to change the estimand.

Nuisance models train on complete timestamp panels before the holdout. HAC uncertainty uses the
declared temporal ordering, and contiguous within-product blocks define the placebo refutation.
`12_model_analysis` reports the causal diagnostics. Trading configuration selection uses
validation backtest Sharpe.

Nothing estimated here competes for that selection. A causal result is not a candidate
configuration: it produces no prediction set, is not eligible for a backtest, and never enters
the funnel. A reader looking for it on the leaderboard will not find it, and that absence is
the design rather than an omission - the two stages answer questions that do not compare.

One consequence for how the result is read: a small or insignificant effect here does not
invalidate the trading strategy, and a large one does not endorse it. The strategy stands or
falls on out-of-sample Sharpe either way. What the estimate changes is what the chapter may
claim about *why* it works.

Prerequisites: `03_financial_features`, `04_model_based_features`, and `05_evaluation`.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# CME 期貨：雙重機器學習

本 Notebook 估計各報酬預測期所設定的處置效果。訓練請求在擬合前，會解析結果變數、處置變數、混雜因子、時序、禁運間隔、干擾模型及安慰劑設計。缺少混雜因子應視為錯誤，而不是改變估計目標的理由。

干擾模型會在留出期間之前的完整時間戳記面板上訓練。HAC 不確定性估計採用已宣告的時間順序；逐商品的連續區塊則定義安慰劑反證。`12_model_analysis` 報告因果診斷。交易設定則依驗證回測夏普比率選擇。

此處的估計結果都不參與交易設定選擇。因果結果不是候選設定：它不產生預測集合、不具備回測資格，也不進入選擇流程。讀者不會在排行榜上找到它；這是設計使然，而非遺漏，因為兩個階段回答的問題無法相互比較。

解讀結果時還有一項影響：此處的效果若小或不顯著，不會推翻交易策略；若效果大，也不會替策略背書。無論哪種情況，策略仍由樣本外夏普比率決定成敗。估計值改變的，是本章對策略*為何*有效所能提出的主張。

先決條件：`03_financial_features`、`04_model_based_features` 與 `05_evaluation`。
<!-- ML4T_ZH_END c0000-p000 -->


## The question, and why this case study in particular has to ask it

The treatment here is `carry_pct` - the term-structure spread between the front contract and
the next one. That is not an arbitrary choice of variable to interrogate. **Carry is the
signal this entire case study trades.** Every model in stages 06 through 10 is, in one form or
another, learning to predict returns from carry and things built out of carry.

So the question this notebook asks is whether the premise underneath all of that holds: does
carry itself move subsequent returns, or does carry merely travel alongside something else
that does? A predictive model does not care - it profits either way, as long as the
association persists. The chapter's explanation of *why* the strategy works cares a great
deal, because an explanation built on the wrong mechanism is wrong even when the strategy is
profitable.

This is also why the answer cannot come from the backtest. A backtest that earns a positive
Sharpe from carry demonstrates that carry predicted returns over that history. It says nothing
about whether carry was the cause, and no amount of out-of-sample performance converts one
into the other.

### What is being adjusted for, and why each one is a candidate explanation

Double machine learning estimates the effect of the treatment after flexibly removing what the
confounders explain of both the treatment and the outcome. It does this by predicting each
from the confounders alone, using models that need not be linear in anything, and then
estimating the effect from what is left over in each. The predictions are cross-fitted, so the
model that residualizes an observation was never fitted on it.

The three confounders are the three rival explanations worth taking seriously here:


<!-- ML4T_ZH_BEGIN c0001-p000 -->
## 要問的問題，以及為何本案例研究必須特別提出

此處的處置變數是 `carry_pct`，即近月合約與次近月合約之間的期限結構價差。選擇檢驗這個變數並非任意。**持有收益是整個案例研究交易的訊號。**第 06 至 10 階段的每個模型，都以某種形式學習從持有收益及其衍生變數預測報酬。

因此，本 Notebook 要問的是其背後前提是否成立：持有收益本身是否影響後續報酬，或只是與真正造成報酬變化的其他因素同行？預測模型不在意這點，只要關聯持續，就能利用它獲利。但本章對策略*為何*有效的解釋卻非常在意，因為即使策略獲利，建立在錯誤機制上的解釋仍是錯的。

這也是為何回測無法給出答案。若回測顯示持有收益策略有正的夏普比率，只能證明在該段歷史中，持有收益能預測報酬；它無法說明持有收益是否為原因。樣本外表現再好，也無法將預測關係轉化為因果關係。

### 調整哪些變數，以及為何每項都是可能的替代解釋

雙重機器學習先彈性地移除混雜因子對處置與結果各自的解釋部分，再估計處置效果。方法是只用混雜因子分別預測處置與結果；所用模型不必對任何變數都呈線性，再從兩者剩餘部分估計效果。預測採交叉擬合，因此對某筆觀測值取殘差的模型，從未用該筆資料擬合。

三項混雜因子，是此處值得認真看待的三種替代解釋：
<!-- ML4T_ZH_END c0001-p000 -->

- **`vol_21d`.** Contracts in backwardation are frequently contracts under supply stress, and
  stress raises volatility. If volatile contracts earn more simply for being volatile, carry
  would look rewarded without being the reason.
- **`momentum_composite`.** Carry and momentum are correlated in futures - a contract in
  sustained backwardation has often been rising. An unadjusted carry effect could be a
  momentum effect wearing carry's label.
- **`carry_rank`.** This is the subtle one, and it is the reason the confounder list is not
  just "the other features". `carry_rank` is a product's position in the cross-section, while
  `carry_pct` is its level. Those come apart: a contract can have a high carry level in a
  period when every contract does, and rank in the middle of its peers. Adjusting for the rank
  asks whether the carry *level* carries information beyond where the product sits relative to
  the others - which is precisely what a cross-sectional strategy is already exploiting.

### Why the standard error is HAC

The uncertainty around the effect is computed with a heteroskedasticity- and
autocorrelation-consistent estimator, because the ordinary one assumes the residuals are
independent and identically scattered and neither holds on a futures panel. Volatility
clusters, so the scatter is larger in some periods than others; and overlapping outcomes tie
neighbouring residuals together, so they carry information about each other. Both inflate the
effective information in the sample relative to what an ordinary standard error assumes, and
the result is a confidence interval that is too narrow and a t-statistic that is too large.
The HAC estimator does not fix the estimate - it corrects what may be claimed about it.


<!-- ML4T_ZH_BEGIN c0001-p001 -->
- **`vol_21d`。**處於逆價差的合約，往往面臨供應壓力，而壓力會提高波動率。若波動較大的合約只是因為波動率高而賺得較多，持有收益看似獲得報酬，卻不是原因。
- **`momentum_composite`。**期貨的持有收益與動能相關；持續處於逆價差的合約，價格通常也已上漲。未經調整的持有收益效果，可能只是披著持有收益名稱的動能效果。
- **`carry_rank`。**這項較細微，也是混雜因子清單不只是「其他特徵」的原因。`carry_rank` 是商品在橫斷面中的位置，`carry_pct` 則是其水準。兩者可能不同：某合約的持有收益水準很高，但若當時所有合約都高，它在同類商品中可能只排中間。調整排名，是要問持有收益*水準*是否提供超出商品相對位置的資訊；而相對位置正是橫斷面策略已利用的內容。

### 為何標準誤使用 HAC

效果的不確定性採異質變異及自相關一致估計量計算，因為一般估計量假設殘差獨立、同分布，但在期貨面板上兩者都不成立。波動率會群聚，所以有些時期的分散程度較大；重疊的結果期間也使相鄰殘差相關，彼此帶有共同資訊。相較於一般標準誤的假設，這兩種現象都會使樣本看似包含過多有效資訊，結果便是信賴區間過窄、t 統計量過大。HAC 估計量不會修正效果估計值，而是修正可據此提出的統計主張。
<!-- ML4T_ZH_END c0001-p001 -->

### Why the label buffer sizes the placebo block here, and not the treatment

The refutation permutes the treatment in contiguous blocks and re-runs the whole estimation,
building a distribution of effects under the hypothesis that the treatment does nothing. The
block has to be long enough to preserve whatever serial dependence the data has, or the
placebo is a weaker opponent than the truth and every p-value looks significant.

Two separate scales create that dependence, and the block spans the longer of them:
`block_size = max(label_buffer, treatment_window)`. The overlapping labels span the label
horizon, and the treatment's own construction window spans itself.

`causal.treatment_window` is 1 for this case study, and the reason is a property of the
construction rather than a judgement. `carry_pct` is
`(c0_price - c1_price) / c0_price * 12`, computed from two prices at the same timestamp.
Nothing rolls, nothing averages, no window is spanned. The label buffer is therefore the
binding scale here: the registered blocks are 5 periods for `fwd_ret_5d` and 21 for
`fwd_ret_21d`, and each row records its own `block_size` beside a `block_size_basis` of
`label_buffer` saying which of the two set it.


<!-- ML4T_ZH_BEGIN c0001-p002 -->
### 為何此處由標籤緩衝長度決定安慰劑區塊，而非處置變數

反證程序會以連續區塊置換處置變數，並重新執行整套估計，建立「處置沒有作用」假說下的效果分布。區塊必須夠長，才能保留資料原有的序列相依性；否則安慰劑檢驗會比真實情況更容易通過，所有 p 值看起來都顯著。

相依性來自兩種不同尺度，區塊長度取兩者較長者：`block_size = max(label_buffer, treatment_window)`。重疊標籤跨越標籤的預測期，處置變數自身的建構視窗則跨越其視窗期間。

本案例研究的 `causal.treatment_window` 為 1，原因是建構方式的性質，而非主觀判斷。`carry_pct` 是由同一時間戳記的兩個價格計算出的 `(c0_price - c1_price) / c0_price * 12`；沒有滾動、沒有取平均，也沒有跨越視窗。因此此處由標籤緩衝長度決定區塊：`fwd_ret_5d` 的登錄區塊為 5 期，`fwd_ret_21d` 為 21 期。每列在 `block_size` 記錄自身區塊長度，並以 `block_size_basis` 的 `label_buffer` 指出是哪一種尺度決定了長度。
<!-- ML4T_ZH_END c0001-p002 -->

**A one-bar construction window is not a claim that the column is serially independent**, and
the two are worth keeping apart. The declared window says how many bars the formula reads;
the column's empirical persistence is a separate quantity and is much larger here.
`12_model_analysis` measures it from the feature panel rather than asserting it, on one row per
product-session because that is the unit the block counts, pooling the autocorrelation within
product with each product demeaned first. Compare each block against the autocorrelation at
that lag: 0.52 at the 5 sessions used for `fwd_ret_5d`, 0.14 at the 21 used for
`fwd_ret_21d`, and indistinguishable from zero by lag 63. The 5-session block leaves real
dependence unpreserved and narrows that label's placebo distribution by some amount neither
notebook quantifies; the 21-session block spans most of it. The narrowing bears mainly on one
of the two refutations here, not equally on both.

The contrast with a rolling treatment is worth holding onto, because it is where this is
usually got wrong. A treatment built from a 252-session window overlaps its neighbours in 251
of them, and a block sized by a 21-day label buffer would shred that overlap; the placebo
distribution narrows, and the p-value collapses toward zero whether or not the effect is
real. That is the case the `max` exists for. The number is declared in `setup.yaml` and
derived from how the column is built, because inferring it from a window list would put a
wrong number behind a right-looking one.

<!-- ML4T_ZH_BEGIN c0001-p003 -->
**建構視窗只有一根價格資料，不代表該欄位在序列上彼此獨立**；兩者必須區分。已宣告視窗說明公式讀取多少根價格資料；欄位在實證上的持續性則是另一個量，此處遠大於一根。`12_model_analysis` 不是直接斷言，而是從特徵面板衡量；每項商品每個交易日取一列，因為這是區塊計數單位，並先對每項商品去平均，再彙整商品內的自相關。將各區塊與相同落後期的自相關比較：`fwd_ret_5d` 所用的 5 個交易日為 0.52，`fwd_ret_21d` 所用的 21 個交易日為 0.14，至落後 63 期時則已無法與零區分。5 日區塊未保留真實存在的部分相依性，使該標籤的安慰劑分布縮窄，但兩份 Notebook 都未量化縮窄程度；21 日區塊則涵蓋大部分相依性。這項縮窄主要影響此處兩項反證中的一項，而非兩項程度相同。

應記住與滾動處置變數的對比，因為這正是常見錯誤所在。由 252 個交易日視窗建構的處置變數，與相鄰觀測值有 251 日重疊；若依 21 日標籤緩衝設定區塊，就會打散這種重疊，縮窄安慰劑分布，無論效果是否真實，都使 p 值趨近零。`max` 正是為處理這種情況而存在。其數值在 `setup.yaml` 宣告，並依欄位建構方式推得；若從視窗清單推斷，就可能在外觀合理的數字背後藏著錯誤數值。
<!-- ML4T_ZH_END c0001-p003 -->


In [1]:
"""Fit the declared CME futures double-machine-learning requests."""

import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    open_study,
    product_universe_table,
)
from case_studies.research import causal_supersedes

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_REDUCTIONS: dict = {}
# Retired by this run: the block-permutation refutation now compares the HAC t-statistic
# rather than the raw effect, so CAUSAL_RUNNER_VERSION moved and every causal identity with
# it. The rows named here hold a p-value computed on the shrunken placebo effects; this run
# supersedes them rather than correcting them, because the statistic is different, not the
# arithmetic. Read out of each registry's current canonical identity per label, 2026-09-10.
SUPERSEDES_CAUSAL: str = '{"fwd_ret_5d": "4abb82b8141c", "fwd_ret_21d": "ac5c1a480d24"}'

## Resolve the estimands

Preview row and fold limits must be named in `PREVIEW_REDUCTIONS`. They enter the computation
identity and cannot be mistaken for canonical causal estimates.

**Missing confounders are an error rather than an invitation to change the estimand**, which
is worth stating as a rule because the alternative is so tempting. Dropping a confounder that
failed to resolve leaves a request that still runs and still returns a number - a number
answering a different question, with no adjustment for the thing that went missing, and
nothing in the result marking it. An estimand is the whole specification: outcome, treatment,
confounder set, timing and embargo together. Changing any part of it produces a different
quantity, not a degraded version of the same one.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 解析估計目標

預覽時對資料列及資料折的限制，必須在 `PREVIEW_REDUCTIONS` 中明確命名。這些限制會進入運算識別資訊，不會被誤認為正式因果估計值。

**缺少混雜因子應視為錯誤，而不是改變估計目標的理由**；這值得明定為規則，因為另一種做法很容易令人心動。刪除解析失敗的混雜因子後，請求仍能執行並傳回數值，但它回答的是另一個問題：缺少對該變數的調整，而結果也不會標出這項缺漏。估計目標是整份規格，包含結果變數、處置變數、混雜因子集合、時序及禁運間隔。改變其中任何一部分，得到的都是不同的量，而非同一量的較差版本。
<!-- ML4T_ZH_END c0004-p000 -->


In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
if EXECUTION_TIER == "preview" and (WORKSPACE is None or not PREVIEW_REDUCTIONS):
    raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [4]:
requests = tuple(
    study.causal(
        method="dml",
        label=label,
        execution_tier=EXECUTION_TIER,
        preview_reductions=PREVIEW_REDUCTIONS,
        supersedes=causal_supersedes(
            study,
            SUPERSEDES_CAUSAL,
            label,
            labels=list(ALL_LABELS),
            execution_tier=EXECUTION_TIER,
        ),
    ).resolve()
    for label in ALL_LABELS
)
request_rows = []
for request in requests:
    computation = request.spec["computation"]
    estimand = computation["estimand"]
    population = computation["analysis_population"]
    request_rows.append(
        {
            "label": request.spec["label"],
            "treatment": estimand["treatment"],
            "outcome": estimand["outcome"],
            "outcome_horizon": estimand["outcome_horizon"],
            "confounders": ", ".join(estimand["confounders"]),
            "analysis_rows": population["n_rows"],
            "analysis_timestamps": population["n_timestamps"],
            "request_hash": request.identity,
        }
    )
request_table = pl.DataFrame(request_rows)
request_table

label,treatment,outcome,outcome_horizon,confounders,analysis_rows,analysis_timestamps,request_hash
str,str,str,str,str,i64,i64,str
"""fwd_ret_5d""","""carry_pct""","""fwd_ret_5d""","""5 days 00:00:00""","""vol_21d, momentum_composite, c…",88633,3098,"""a50a55ef32e8"""
"""fwd_ret_21d""","""carry_pct""","""fwd_ret_21d""","""21 days 00:00:00""","""vol_21d, momentum_composite, c…",88105,3082,"""8d7cc0d7a423"""


## Execute and verify restart

The shared causal runner uses deterministic seeds and persists one immutable result per resolved
request. Reopening the same request must return the same identity and a complete result.

The restart check runs the request twice and requires the same hash. That is cheap here
because the second call is served from the registry, and it is worth doing because a causal
estimate has no external reference to be checked against. A predictive model can be caught by
out-of-sample performance; an effect estimate cannot, so reproducibility is most of what is
available. An estimate that moved between two runs of the same specification would mean the
seed does not control everything the fit depends on, and the number would be one draw from a
distribution nobody characterised.

`SUPERSEDES_CAUSAL` above is how a re-run names the identity it retires, per label. A refit
under a changed specification produces a second current identity for the same label, and
`CausalResult.one` resolves a label to exactly one - so the run has to say which it replaces,
and the retired estimate stays in the registry rather than being deleted.

<!-- ML4T_ZH_BEGIN c0007-p000 -->
## 執行與驗證重新啟動

共用因果執行器使用確定性種子，並對每個解析後請求保存一份不可變的結果。重新開啟相同請求，必須傳回相同識別資訊及完整結果。

重新啟動檢查會執行同一請求兩次，要求兩次雜湊相同。此處第二次呼叫直接由登錄表傳回，因此成本很低；又因因果估計值沒有外部基準可核對，這項檢查值得做。預測模型可由樣本外表現發現問題，效果估計值則無法，因此可重現性幾乎是目前可用的主要檢查。若同一規格執行兩次得到不同估計值，就表示種子未控制擬合所依賴的全部因素，而該數值只是一個分布中的一次抽樣，該分布卻未經描述。

上方 `SUPERSEDES_CAUSAL` 表示重跑時逐標籤指明要淘汰的識別資訊。變更規格後重新擬合，會為同一標籤產生第二個目前有效的識別資訊，而 `CausalResult.one` 會將標籤解析至唯一一個識別資訊，因此執行時必須說明取代哪個版本；被淘汰的估計值會保留在登錄表中，而非刪除。
<!-- ML4T_ZH_END c0007-p000 -->


In [5]:
results = []
for request in requests:
    label = request.spec["label"]
    result = request.run()
    restarted = request.run()
    if not result.complete or restarted.hash != result.hash:
        raise RuntimeError(f"causal request for {label} did not persist completely")
    results.append(
        {
            "label": label,
            "execution_tier": result.execution_tier,
            "complete": result.complete,
            "causal_hash": result.hash,
        }
    )

In [6]:
pl.DataFrame(results).sort("label")

label,execution_tier,complete,causal_hash
str,str,bool,str
"""fwd_ret_21d""","""canonical""",true,"""8d7cc0d7a423"""
"""fwd_ret_5d""","""canonical""",true,"""a50a55ef32e8"""
